# ✈️ Airline Passenger Satisfaction - Exploratory Data Analysis (EDA)

## 1. Problem Statement
This notebook conducts an in-depth exploratory analysis on the **Airline Passenger Satisfaction** dataset.
The objective is to understand key drivers of customer sentiment, identify friction points in the travel journey, and formulate predictive features for classification.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)

# Load dataset
df = pd.read_csv("../data/train.csv")
print(f"Dataset Dimensions: {df.shape[0]:,} rows, {df.shape[1]} columns")
df.head()

## 2. Dataset Schema & Missing Value Check

In [ ]:
# Drop technical index columns if present
drop_cols = [c for c in ["Unnamed: 0", "id"] if c in df.columns]
df = df.drop(columns=drop_cols)

# Check data types and missing values
missing = df.isnull().sum()
missing = missing[missing > 0]
print("Missing value summary:")
print(missing)

df.describe().T

## 3. Target Distribution: Passenger Satisfaction
Let us examine class balance between `satisfied` and `neutral or dissatisfied` passengers.

In [ ]:
plt.figure(figsize=(6, 4))
ax = sns.countplot(data=df, x="satisfaction", palette=["#ef4444", "#10b981"])
plt.title("Passenger Satisfaction Distribution")
plt.xlabel("Satisfaction Status")
plt.ylabel("Count")
total = len(df)
for p in ax.patches:
    percentage = f"{100 * p.get_height() / total:.1f}%"
    ax.annotate(percentage, (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                ha='center', va='center', color='white', fontweight='bold')
plt.show()

## 4. Demographic & Flight Segment Analysis
How does customer satisfaction vary across Cabin Class, Customer Loyalty, and Purpose of Travel?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.countplot(data=df, x="Class", hue="satisfaction", ax=axes[0], palette="Set2")
axes[0].set_title("Satisfaction by Travel Class")

sns.countplot(data=df, x="Customer Type", hue="satisfaction", ax=axes[1], palette="Set2")
axes[1].set_title("Satisfaction by Loyalty Program")

sns.countplot(data=df, x="Type of Travel", hue="satisfaction", ax=axes[2], palette="Set2")
axes[2].set_title("Satisfaction by Travel Purpose")

plt.tight_layout()
plt.show()

## 5. In-Flight Service Touchpoints vs Satisfaction
Comparing the average ratings across 14 service touchpoints between satisfied and dissatisfied passengers.

In [ ]:
service_cols = [
    "Inflight wifi service", "Departure/Arrival time convenient",
    "Ease of Online booking", "Gate location", "Food and drink",
    "Online boarding", "Seat comfort", "Inflight entertainment",
    "On-board service", "Leg room service", "Baggage handling",
    "Checkin service", "Inflight service", "Cleanliness"
]

avg_ratings = df.groupby("satisfaction")[service_cols].mean().T

avg_ratings.plot(kind="barh", figsize=(12, 8), color=["#f87171", "#34d399"])
plt.title("Service Touchpoints: Average Rating by Passenger Sentiment")
plt.xlabel("Average Score (0-5)")
plt.ylabel("Service Touchpoint")
plt.legend(title="Status")
plt.tight_layout()
plt.show()

## 6. Flight Delays & Satisfaction Impact

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df, x="satisfaction", y="Arrival Delay in Minutes", ax=ax, showfliers=False, palette="pastel")
plt.title("Arrival Delay Distribution by Passenger Satisfaction (Without Outliers)")
plt.show()

## 7. Numerical Feature Correlation Matrix

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns
corr = df[num_cols].corr()

plt.figure(figsize=(14, 11))
sns.heatmap(corr, annot=False, cmap="coolwarm", center=0, linewidths=0.5)
plt.title("Correlation Matrix of Survey Features")
plt.tight_layout()
plt.show()

## 8. Summary of Findings
1. **Dominant Drivers**: `Online boarding`, `Inflight wifi service`, `Inflight entertainment`, and `Seat comfort` exhibit the widest satisfaction disparity.
2. **Cabin Class Disparity**: Business Class flyers have an overwhelming rate of satisfaction (~70%), whereas Eco flyers represent the vast majority of dissatisfied ratings.
3. **Missing Data**: `Arrival Delay in Minutes` contains occasional missing values, which can be imputed with median delay.
4. **Modeling Approach**: Tree-based ensembles (`RandomForestClassifier`) will capture complex non-linear combinations of service ratings and flight distance effectively.